# Cycle 1 — Data Exploration: `premier_league_matches.csv`

**Project:** Football Predictor — Match Outcome Prediction (Win / Draw / Loss)  
**Dataset:** `premier_league_matches.csv`  
**Source:** football-data.co.uk (season CSVs from 2000–2020), preprocessed and merged via the Kaggle notebook *"This is Best English Premier League Notebook"* by daniyalatta.  
**Kaggle Reference:** https://www.kaggle.com/code/daniyalatta/this-is-best-english-premier-league-notebook

---

## Purpose of this Notebook

This notebook is dedicated to **exploratory data analysis (EDA)** of the primary dataset used for match outcome prediction. The goal is to:
- Understand the structure, shape, and data types
- Identify what each column represents
- Spot missing values, data quality issues, and anomalies
- Document observations and issues that need to be addressed in preprocessing and feature engineering

This notebook does **not** train any models. It is purely for understanding the data.

---
## Key Findings at a Glance

These are the most important things to know about this dataset before doing anything with it.

### 1. The target variable is broken
The `FTR` column — which should contain H (Home Win), D (Draw), A (Away Win) — only contains `H` and `NH` (Not Home). The original Kaggle author converted this into a binary problem. This means we **cannot use `FTR` directly**. Instead, we reconstruct the correct 3-class target from `FTHG` (home goals) and `FTAG` (away goals): if home goals > away goals it is a Home Win, if less it is an Away Win, if equal it is a Draw. This is done in preprocessing.

### 2. No missing values
All 40 columns are complete across all 6,840 rows. This is because the dataset was already cleaned before publishing on Kaggle. No imputation is needed.

### 3. Early season matches have incomplete form history
The columns `HM1`–`HM5` and `AM1`–`AM5` store each team's last 5 results. For early matchweeks (1–5), a team has not yet played 5 games, so these columns contain `M` (Missing). This is expected behaviour — `M` must be encoded as a neutral value (0) in preprocessing rather than treated as an error.

### 4. Home advantage is measurable in the data
The average home goals scored is 1.53 vs 1.13 for away goals. Home teams win more often (3,176 Home Wins vs 1,902 Away Wins before binary conversion). This is a known real-world effect and a key insight for the report.

### 5. Rich form features are already engineered
Unlike raw datasets, this one already includes rolling form features: points per game (`HTP`/`ATP`), goal difference (`HTGD`/`ATGD`), win/loss streaks, and form points. This means minimal feature engineering is needed — the main task is preprocessing and model training.

### 6. Features have different scales
Some columns like `HTGS` (cumulative season goals) can reach values over 100, while streak columns are binary (0 or 1). This means **normalisation is required** for models like Logistic Regression. Tree-based models (Random Forest, XGBoost) do not need this.

---

---
## Cell 1 — Load the Dataset

**What it does:** Imports the pandas library and loads `premier_league_matches.csv` into a DataFrame. Then prints the shape and displays the first 5 rows.

**Why:** The first step in any data exploration is to load the data and get a high-level view of its dimensions and structure.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/premier_league_matches.csv')
print(df.shape)
df.head()

### Output
- Shape: **(6840, 40)** — 6,840 matches, 40 columns
- First 5 rows show matches starting from 19/08/2000

### Observations
- The dataset covers Premier League matches starting from the 2000/01 season
- 6,840 rows gives us a large, solid dataset for training
- The `FTR` column (target variable) shows values of `H` and `NH` — **this is a problem** (see Cell 3)
- There is an `Unnamed: 0` column which is just a row index — not useful as a feature

### Issues to Fix in Preprocessing
- Drop `Unnamed: 0` — it is a redundant index column
- Investigate `FTR` column values — `NH` is not a standard result label

---
## Cell 2 — Column Names

**What it does:** Prints all 40 column names as a list.

**Why:** The `df.head()` output truncates columns with `...`. This cell ensures we can see every column name.

In [ ]:
print(df.columns.tolist())

### Output
```
['Unnamed: 0', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP',
 'HM1', 'HM2', 'HM3', 'HM4', 'HM5',
 'AM1', 'AM2', 'AM3', 'AM4', 'AM5',
 'MW', 'HTFormPtsStr', 'ATFormPtsStr', 'HTFormPts', 'ATFormPts',
 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5',
 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5',
 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']
```

### Column Reference Guide

| Column | Description |
|---|---|
| `Unnamed: 0` | Redundant row index — drop this |
| `Date` | Match date (dd/mm/yy) |
| `HomeTeam` / `AwayTeam` | Team names |
| `FTHG` | Full Time Home Team Goals |
| `FTAG` | Full Time Away Team Goals |
| `FTR` | Full Time Result — **target variable** (broken, see Cell 3) |
| `HTGS` / `ATGS` | Home/Away Team Goals Scored (cumulative season total) |
| `HTGC` / `ATGC` | Home/Away Team Goals Conceded (cumulative season total) |
| `HTP` / `ATP` | Home/Away Team Points per game ratio |
| `HM1–HM5` | Home team's last 5 match results (W/D/L/M) |
| `AM1–AM5` | Away team's last 5 match results (W/D/L/M) |
| `MW` | Matchweek number (1–38) |
| `HTFormPtsStr` / `ATFormPtsStr` | Form as a string e.g. "WWDLW" |
| `HTFormPts` / `ATFormPts` | Form points (numeric) from last 5 matches |
| `HTWinStreak3/5` | 1 if home team is on a 3/5 game win streak, else 0 |
| `HTLossStreak3/5` | 1 if home team is on a 3/5 game loss streak, else 0 |
| `ATWinStreak3/5` | 1 if away team is on a 3/5 game win streak, else 0 |
| `ATLossStreak3/5` | 1 if away team is on a 3/5 game loss streak, else 0 |
| `HTGD` / `ATGD` | Home/Away Team Goal Difference (normalised) |
| `DiffPts` | Difference in points per game between the two teams |
| `DiffFormPts` | Difference in form points between the two teams |

### Observations
- The dataset already has rich pre-engineered form features (streaks, form points, goal difference)
- `FTHG` and `FTAG` are the raw goals — these are the ground truth for reconstructing the real `FTR`
- `HTFormPtsStr` / `ATFormPtsStr` are string representations — not useful as model features directly, will be dropped

### Notes for Preprocessing
- Drop: `Unnamed: 0`, `HTFormPtsStr`, `ATFormPtsStr`, `FTHG`, `FTAG` (after using them to fix `FTR`)
- Keep: all numeric form features as model input features

---
## Cell 3 — Data Types

**What it does:** Prints the data type of every column.

**Why:** Knowing the data types tells us which columns need encoding (object → numeric) before they can be used in a machine learning model.

In [ ]:
print(df.dtypes)

### Output
```
Unnamed: 0       int64
Date             object
HomeTeam         object
AwayTeam         object
FTHG             int64
FTAG             int64
FTR              object
HTGS             int64
ATGS             int64
HTGC             int64
ATGC             int64
HTP              float64
ATP              float64
HM1–HM5          object
AM1–AM5          object
MW               float64
HTFormPtsStr     object
ATFormPtsStr     object
HTFormPts        int64
ATFormPts        int64
HTWinStreak3–5   int64
HTLossStreak3–5  int64
ATWinStreak3–5   int64
ATLossStreak3–5  int64
HTGD             float64
ATGD             float64
DiffPts          float64
DiffFormPts      float64
```

### Observations
- **Object columns that need encoding:** `Date`, `HomeTeam`, `AwayTeam`, `FTR`, `HM1–HM5`, `AM1–AM5`, `HTFormPtsStr`, `ATFormPtsStr`
- **Numeric columns (ready to use):** all `int64` and `float64` columns
- `MW` is `float64` — should be `int64`, minor issue

### Notes for Preprocessing
- `HM1–HM5` and `AM1–AM5`: encode W=3, D=1, L=0, M=0 (or use one-hot encoding)
- `HomeTeam` / `AwayTeam`: label encode or one-hot encode
- `Date`: convert to datetime, may extract season or month as a feature, then drop
- `FTR`: reconstruct from `FTHG`/`FTAG` (see Cell 4)

---
## Cell 4 — Target Variable Distribution

**What it does:** Counts how many times each result value appears in the `FTR` column.

**Why:** The target variable is what the model is trying to predict. We need to check its distribution for class imbalance and also verify it contains the correct values (H, D, A).

In [ ]:
print("Target variable distribution:")
print(df['FTR'].value_counts())

### Output
```
FTR
NH    3664
H     3176
Name: count, dtype: int64
```

### CRITICAL ISSUE — Binary Target Instead of 3-Class

The `FTR` column only contains two values:
- `H` — Home Win
- `NH` — Not Home (which groups **Draws and Away Wins together**)

This means the original Kaggle notebook converted this into a **binary classification problem** (Home Win vs Not Home Win). This is **not suitable** for our project, which requires 3-class prediction: **H (Home Win), D (Draw), A (Away Win)**.

### Fix
We can reconstruct the correct 3-class `FTR` from the `FTHG` and `FTAG` columns which contain the actual goals scored:
- `FTHG > FTAG` → **H** (Home Win)
- `FTHG < FTAG` → **A** (Away Win)
- `FTHG == FTAG` → **D** (Draw)

This will be done in the preprocessing notebook.

### Notes for Report
- The source dataset had a pre-existing limitation: the `FTR` column was binarised (H vs NH) by the original author
- This required us to reconstruct the true 3-class target variable from raw goal data
- This is a good example of why raw data must always be inspected before use

---
## Cell 5 — Missing Values

**What it does:** Counts the number of null/missing values in every column.

**Why:** Missing values must be identified before training — most ML models cannot handle NaN values and will throw errors or produce wrong results.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

### Output
All 40 columns show **0 missing values**.

### Observations
- The dataset is complete — no imputation or missing value handling is required
- This is because the dataset was already preprocessed by the original Kaggle author before publishing

### Notes
- No action needed for missing values in preprocessing

---
## Cell 6 — Last 5 Match Result Values

**What it does:** Checks the unique values in `HM1` (home team's most recent match result).

**Why:** We need to know every possible value in the form columns to handle them correctly during encoding.

In [ ]:
print("Unique values in HM1 (home team last match result):")
print(df['HM1'].value_counts())

### Output
```
HM1
L    2700
W    2226
D    1734
M     180
Name: count, dtype: int64
```

### Observations
- `W`, `D`, `L` are the expected result values
- `M` = **Missing** — used for early season matches (Matchweek 1–5) when a team hasn't played enough games to have 5 previous results
- `M` appears 180 times in `HM1` — this is expected behaviour, not a data error
- The same pattern applies to `HM2–HM5` and `AM1–AM5`

### Notes for Preprocessing
- When encoding `HM1–HM5` and `AM1–AM5`, treat `M` as a neutral value (e.g. encode as 0)
- Suggested encoding: W=3, D=1, L=0, M=0

---
## Cell 7 — Summary Statistics

**What it does:** Produces descriptive statistics (count, mean, std, min, max, quartiles) for all numeric columns.

**Why:** Summary statistics reveal the range, spread, and central tendency of each feature — helping us spot outliers and understand the scale of data.

In [ ]:
df.describe()

### Key Observations

**Goals (`FTHG`, `FTAG`):**
- Average home goals: **1.53**, average away goals: **1.13**
- This confirms the well-known **home advantage** effect in football
- Maximum home goals: **9**, maximum away goals: **7** — these are real outlier matches
- For the report: home advantage is a real, measurable effect in the data

**Points per game (`HTP`, `ATP`):**
- Ranges from 0 to ~2.76 (maximum possible is 3.0 — winning every game)
- These are normalised ratios, not raw points — good for cross-season comparison

**Matchweek (`MW`):**
- Ranges from 1 to 38 — covers a full Premier League season

**Streak columns (`HTWinStreak3/5`, `HTLossStreak3/5`, etc.):**
- Almost all values are 0 — win/loss streaks of 3 or 5 consecutive games are rare
- Max is 1 (binary flags — either on a streak or not)

**Goal Difference (`HTGD`, `ATGD`):**
- Centered around 0 (mean ≈ -0.01 and 0.01) — balanced across home and away
- `DiffPts` and `DiffFormPts` are also centered at 0 — symmetric between home and away teams

### Notes for Preprocessing
- Numeric features have different scales (e.g. `HTGS` goes up to 102, while streak columns are 0/1)
- **Normalisation or standardisation** will be needed before training models like Logistic Regression
- Tree-based models (Random Forest, XGBoost) are scale-invariant and do not require normalisation

---
## Summary of Findings

| # | Finding | Severity | Action Required |
|---|---|---|---|
| 1 | `FTR` is binary (H/NH) instead of 3-class (H/D/A) | **Critical** | Reconstruct from `FTHG` and `FTAG` in preprocessing |
| 2 | `Unnamed: 0` is a redundant index column | Low | Drop in preprocessing |
| 3 | `HM1–HM5`, `AM1–AM5` contain `M` (missing) values | Medium | Encode `M` as 0 in preprocessing |
| 4 | `HTFormPtsStr`, `ATFormPtsStr` are string representations | Low | Drop — numeric form columns already exist |
| 5 | Features have varying scales | Medium | Apply StandardScaler for Logistic Regression |
| 6 | No missing values | — | No action needed |
| 7 | Home advantage confirmed (avg FTHG 1.53 vs FTAG 1.13) | — | Useful observation for report |

---

## Next Steps
1. Create `cycle1_exploration_skysports_match_stats.ipynb` — explore the second dataset
2. After both explorations, create `cycle1_preprocessing.ipynb` — clean and prepare data for modelling
3. Then `cycle1_modelling.ipynb` — train and evaluate models